# Prompt 12: Multi-Modal Prompting

1. Image extraction with a structured schema
2. Compare image-first vs text-first placement
3. Chart reading prompt
4. Code generation with constraints
5. Exercises: screenshot-agent sketch; audio prompt with initial_prompt

Deps: `pip install anthropic openai pillow`

In [ ]:
import os, io, base64
from PIL import Image, ImageDraw

def make_invoice():
    img = Image.new('RGB', (620, 380), 'white')
    d = ImageDraw.Draw(img)
    for y, line in enumerate([
        'NovaCore Industries', 'Invoice #2026-04-0042', '',
        'Widget Pro x 3    @ $25.00  = $75.00',
        'Adapter Cable x 5 @ $4.50   = $22.50',
        'Shipping                    = $12.00',
        '----------------------------', 'Total                       = $109.50',
        '', 'Due: 2026-05-10'
    ], start=0):
        d.text((30, 25 + y*30), line, fill='black')
    return img

img = make_invoice()
buf = io.BytesIO(); img.save(buf, 'PNG')
b64 = base64.b64encode(buf.getvalue()).decode()

print('image ready')
display(img)

## 1. Image-First vs Text-First Placement (Claude)

In [ ]:
if not os.getenv('ANTHROPIC_API_KEY'):
    print('Skipping Claude sections (no ANTHROPIC_API_KEY).')
else:
    from anthropic import Anthropic
    client = Anthropic()
    img_block = {'type':'image','source':{'type':'base64','media_type':'image/png','data':b64}}
    task = 'Return JSON: {"vendor":string,"invoice_number":string,"total_usd":number,"due":"YYYY-MM-DD"|null}. JSON only.'

    r1 = client.messages.create(model='claude-sonnet-4-6', max_tokens=300,
        messages=[{'role':'user','content':[img_block, {'type':'text','text': task}]}])
    print('--- image-first ---\n', r1.content[0].text)

    r2 = client.messages.create(model='claude-sonnet-4-6', max_tokens=300,
        messages=[{'role':'user','content':[{'type':'text','text': task}, img_block]}])
    print('\n--- text-first ---\n', r2.content[0].text)

## 2. Confidence-Annotated Extraction

In [ ]:
schema_prompt = '''For each field, return both the value and a confidence level ("high", "medium", "low").

Return JSON:
{
  "vendor": {"value": string, "confidence": string},
  "total_usd": {"value": number, "confidence": string},
  "due": {"value": "YYYY-MM-DD"|null, "confidence": string}
}

Use "high" if clearly visible, "medium" if partially obscured, "low" if inferred or unclear.'''

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=400,
        messages=[{'role':'user','content':[img_block, {'type':'text','text': schema_prompt}]}])
    print(r.content[0].text)

## 3. Chart Reading Prompt

In [ ]:
# Generate a simple bar chart and ask the model to extract its data.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
labels = ['Q1','Q2','Q3','Q4']
vals = [3.2, 4.1, 5.7, 4.9]
ax.bar(labels, vals, color='tab:blue')
ax.set_title('Quarterly Revenue (millions USD)')
ax.set_ylabel('Revenue (M USD)')
ax.set_ylim(0, 7)
for i, v in enumerate(vals): ax.text(i, v + 0.2, f'${v}M', ha='center')
fig.tight_layout()

buf2 = io.BytesIO(); fig.savefig(buf2, format='PNG'); b64_chart = base64.b64encode(buf2.getvalue()).decode()
plt.close(fig)

if os.getenv('ANTHROPIC_API_KEY'):
    from anthropic import Anthropic
    chart_block = {'type':'image','source':{'type':'base64','media_type':'image/png','data':b64_chart}}
    chart_prompt = '''Extract the chart data as JSON:
{"title":str,"y_axis_label":str,"series":[{"x":str,"y":number}]}
Plus a one-sentence description in "summary". Return ONLY JSON.'''
    r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=400,
        messages=[{'role':'user','content':[chart_block, {'type':'text','text': chart_prompt}]}])
    print(r.content[0].text)

## 4. Code Generation with Constraints

In [ ]:
sys = '''You are a Python engineer. Requirements:
- Python 3.11
- Type hints on all signatures
- `ruff`-compliant style
- No external dependencies beyond the standard library
- Include `pytest` tests covering: normal, empty input, one-item, duplicates.
Return ONLY code. Put tests in the same file under `if __name__ == "__main__":` or separate `test_*` functions.'''

user = 'Write `most_frequent(xs: list[int]) -> int | None` that returns the most common element. On tie, return the smallest.'

def call(system, user_text, max_tokens=600):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens, temperature=0,
            system=system, messages=[{'role':'user','content':user_text}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=0,
            messages=[{'role':'system','content':system},{'role':'user','content':user_text}])
        return r.choices[0].message.content
    return '[no provider]'

print(call(sys, user))

## 5. Exercise: Screenshot Agent Sketch

Build a tiny loop:
1. Model sees a screenshot.
2. Outputs ONE of: `click(x,y)`, `type('...')`, `scroll(direction)`, `wait`.
3. Mock "execute" the action and take a new screenshot.
4. Repeat until done.

Start with a synthetic UI you draw yourself (a form with buttons). Verify the agent reliably navigates.

## 6. Exercise: Audio Initial Prompt for Domain Terminology

If you have Whisper installed:
```python
w = whisper.load_model('base')
no_hint = w.transcribe('medical_audio.wav')
with_hint = w.transcribe('medical_audio.wav',
    initial_prompt='This is a cardiology consultation. Terms: atrial fibrillation, beta blocker, echocardiogram.')
```

Compare. Domain glossary via `initial_prompt` often fixes terminology transcription errors.

## 7. Exercise: Vision Hallucination Audit

Show the model an image that does NOT contain a vendor name (e.g., a blank receipt or a random photo). Does it fabricate one, admit it can't find one, or return null?

Then add to the prompt: "If the vendor is not clearly visible, return null and set confidence='low'. Do not infer." Measure the fabrication-rate delta. This is the prompt pattern that cuts most vision hallucinations.

## 8. Module 06 Wrap

You've built:

- A mental model of prompt engineering as a discipline.
- Anatomy of a production prompt.
- Zero/few-shot heuristics + CoT / reasoning patterns.
- Persona, structured output, and template craft.
- Advanced techniques (ReAct, self-refine, decompose).
- Prompt optimization + testing + registry.
- Multi-modal adaptations.

**Next module:** `07-data-engineering-for-ai/` — the pipelines that feed your prompts their context.